# Cooking, gate gfail with a second trigger - Qwen3-1.7B, 4B, 8B
Plays `grids/cooking_fail2.yaml`: `gfailidle4` (think when the game refused the last command, or after 4 turns without thinking) and `gfailstuck4` (the same, or when the last 4 commands are at most 2 different ones), prompt `CoT+nonempty`, cooking levels 1-3-5-7-9, seeds 1-10, for the three Qwen3 sizes one after the other (see the grid's comment and THESIS_LOG.md, 2026-11-15). 300 games.
The 8B uses its fp8 KV cache here, as its cooking games already played.
These two gates have only been tested with a scripted model: look at the first games (`limit = 4`) before leaving the whole grid running.

# Setup
Clones the repository (only the folders needed), installs TextWorld and vLLM and moves into `Code/Benchmark`. Locally it does nothing.
The first time on a new Colab machine the installation takes a few minutes: restart the kernel after it (and then run all the cells), because it can update torch.

In [1]:
import os
import sys
import shutil
import subprocess

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    repo_url = "https://github.com/UPROOT01/LLMs-Play-Textual-Games.git"
    repo_dir = "/content/LLMs-Play-Textual-Games"
    if not os.path.isfile(f"{repo_dir}/Code/Benchmark/run.py"):
        shutil.rmtree(repo_dir, ignore_errors=True) # leftover of a failed clone, or a clone without the benchmark
        # sparse clone: only the folders needed (the whole Games folder is >1 GB)
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", repo_url, repo_dir], check=True)
    else:
        # the latest code and the latest results pushed to git (they decide what is skipped, with Drive)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    subprocess.run(["git", "-C", repo_dir, "sparse-checkout", "add", "Code/Benchmark",
                    "Code/Games/cooking", "Code/Games/coin_collector", "Code/Games/treasure_hunter"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "textworld", "vllm==0.31.0", "pyyaml"], check=True)
    os.chdir(f"{repo_dir}/Code/Benchmark")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

Working directory: /content/LLMs-Play-Textual-Games/Code/Benchmark


In [2]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import torch

assert torch.cuda.is_available(), "No GPU available: connect to a Colab runtime with a GPU (e.g. L4)"
! nvidia-smi

Sun Nov 15 17:13:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   39C    P8             14W /   72W |       3MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# What to run
The grid and the models are fixed for this notebook. `limit`: at most this many games per model (`None`: all; e.g. `4` for a first look).
`n_parallel = "auto"`: as many games at once as fit in the GPU memory (see `engine.ContextBudget`).

The first cell below prints, for each results folder, how many games of the grid it already holds: the games found in Drive **or** in the git clone are skipped. Results played on another machine are seen only once they are committed and pushed.

In [4]:
from records import LEGACY_RESULTS
from run import played_by_folder, summary
from run import load_grid, run_grid

grid = load_grid("grids/cooking_fail2.yaml")
models = None
limit = None
n_parallel = "auto"

# Where the games already played are looked for: a game found in ANY of these folders is not played again.
#   - Drive: the games played on Colab (new games are saved here)
#   - the git clone (Code/Benchmark/results): the games committed and pushed from anywhere (the remote machine, other
#     Colab runs copied into the repo). The setup cell pulls before this, so it is the latest pushed state.
if IN_COLAB:
    assert os.path.isdir("/content/drive/MyDrive"), "Google Drive is not mounted: run the cell above"
    results = "/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results"
else:
    results = "results"
read = [LEGACY_RESULTS] if os.path.abspath(LEGACY_RESULTS) != os.path.abspath(results) else []
print("New games saved in:", results)
print("Also read:", read, "\n")
played_by_folder(grid, [results, *read], models)
print()
summary(grid, results, read, models)

New games saved in: /content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results
Also read: ['/content/LLMs-Play-Textual-Games/Code/Benchmark/results'] 

/content/drive/MyDrive/LLMs-Play-Textual-Games/Benchmark/results: 0 games of the grid, 0 only here
/content/LLMs-Play-Textual-Games/Code/Benchmark/results: 0 games of the grid, 0 only here
played in at least one folder: 0 of 300

model                                          mode  played  left  ~Mtokens left
Qwen/Qwen3-1.7B                              gfailidle4       0    50               
Qwen/Qwen3-1.7B                              gfailstuck4       0    50               
Qwen/Qwen3-4B                                gfailidle4       0    50               
Qwen/Qwen3-4B                                gfailstuck4       0    50               
Qwen/Qwen3-8B-FP8                            gfailidle4       0    50               
Qwen/Qwen3-8B-FP8                            gfailstuck4       0    50               
total: 0 played, 3

# Run
Every game is saved on Drive as soon as it ends. Stopping the cell is safe: the games being played are lost, the others are kept; run the notebook again to resume.
If loading the second model fails because the GPU memory is full, restart the runtime and run everything again: the first model's games are skipped.

In [5]:
run_grid(grid, results, read, models=models, limit=limit, n_parallel=n_parallel)
summary(grid, results, read, models)

Qwen/Qwen3-1.7B: 100 games to play


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

... (7 outputs left out)

Qwen/Qwen3-1.7B: KV cache of 143616 tokens


12 games running, 35 waiting, 53 ended; since the start 17.5 games/hour, 7794 tokens/min; last 10 min 54.2 games/hour, 5325 tokens/min; KV cache: target 92% (games started on the measured usage), actual 89% on average since the start, 81% in the last 10 min (peak 100%); 2 requests paused (preempted) in the last 10 min, 156 since the start

('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailstuck4', 7, 1): steps 100, score 0/11, 24 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailstuck4', 5, 1): steps 100, score 0/8, 24 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailidle4', 7, 2): loop, steps 28, score 1/11, 70 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailidle4', 5, 4): steps 10, score 1/8, 76 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailstuck4', 1, 5): steps 12, score 1/5, 174 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailstuck4', 9, 1): steps 30, score 1/11, 406 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailidle4', 5, 5): loop, steps 36, score 0/8, 209 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailidle4', 5, 2): loop, steps 45, score 0/8, 439 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailidle4', 3, 5): steps 13, score 1/11, 331 s
('Qwen/Qwen3-1.7B', 'cooking', 'CoT+nonempty', 'gfailidle4', 1, 5): steps 15, score 2/5, 534 s
('Qwen/Qwen3-1.7B', 'cooki

KeyboardInterrupt: 

# Quick look
Per model and mode, from Drive and git together: games, average normalized score, thinking turns and thinking tokens per game. The modes already played on the same levels and seeds (0, e1, er4, lr4, r4, gfail, lgfail, grand25, grand50) are in the second table.

In [ ]:
from records import load_table

table = load_table(results, *read)
table["level"] = table.level.astype(str)
same = (table.game.isin(grid["games"]) & table.model.isin(models or grid["models"]) & table.seed.isin(grid["seeds"])
        & table.level.isin([str(level) for level in grid["games"]["cooking"]]))
per_game = dict(games=("seed", "size"), score=("norm_score", "mean"),
                thinks=("tokens_think_turns", "mean"), kthink=("tokens_generated_thinking", lambda t: t.mean() / 1000))
display(table[same & table["mode"].isin(grid["modes"]) & table.prompt.isin(grid["prompts"])]
        .groupby(["model", "mode"]).agg(**per_game).round(2))
table[same & table["mode"].isin(["0", "e1", "er4", "lr4", "r4", "gfail", "lgfail", "grand25", "grand50"]) & table.prompt.isin(["CoT", "CoT+nonempty"])] \
    .groupby(["model", "prompt", "mode"]).agg(**per_game).round(2)